### Imports & Configrations

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.sql.window import Window


SILVER_TABELS = {
    "customers":       "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/customers",
    "orders":          "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/orders",
    "order_items":     "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/order_items",
    "products":        "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/products",
    "product_reviews": "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/product_reviews"
}


def read_silver(table_name: str) -> DataFrame:
    """Read a Delta table from the Silver lakehouse"""
    path = SILVER_TABELS[table_name]
    return spark.read.format("delta").load(path)

def write_to_gold(df: DataFrame, table_name: str):
    """Write to Gold """
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)
    print(f" {table_name} → {df.count()} rows written to Gold")



StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

### Reading the data

In [ ]:
silver_customers       = read_silver("customers")
silver_orders          = read_silver("orders")
silver_order_items     = read_silver("order_items")
silver_products        = read_silver("products")
silver_product_reviews = read_silver("product_reviews")

print(f"Customers:       {silver_customers.count():,}")
print(f"Orders:          {silver_orders.count():,}")
print(f"Order Items:     {silver_order_items.count():,}")
print(f"Products:        {silver_products.count():,}")
print(f"Product Reviews: {silver_product_reviews.count():,}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### DIM_DATE

In [ ]:
df_dim_date = (
    spark.sql("""
        SELECT explode(sequence(
            to_date('2021-08-31'), 
            to_date('2025-08-30'), 
            interval 1 day
        )) AS full_date
    """)
    .withColumn("date_key", F.date_format(F.col("full_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_of_week", F.dayofweek("full_date"))  # 1=Sun, 7=Sat
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", 
        F.when(F.dayofweek("full_date").isin(1, 7), F.lit(True))
         .otherwise(F.lit(False))
    )
    .select(
        "date_key", "full_date", "year", "quarter",
        "month_number", "month_name", "day_of_week", 
        "day_name", "is_weekend"
    )
)

print(f"Raw date rows: {df_dim_date.count()}")
write_to_gold(df_dim_date, "dim_date")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### DIM_CUSTOMER

In [ ]:
w = Window.orderBy("customer_id")

df_dim_customer = (silver_customers
    .withColumn("customer_key", F.row_number().over(w))
    .withColumn("signup_date", F.col("start_date"))
    .withColumn("is_current",
        F.when(F.col("end_date").isNull(), F.lit(True))
         .otherwise(F.lit(False))
    )
    .select(
        "customer_key",
        "customer_id",
        "name",
        "email",
        "gender",
        "country",
        "signup_date",
        "end_date",
        "is_current"
    )
)

#  Add unknown customers with -1 
unknown_customer = spark.sql("""
    SELECT 
        CAST(-1 AS INT) AS customer_key,
        CAST(-1 AS INT) AS customer_id,
        'Unknown' AS name,
        'unknown' AS email,
        'Unknown' AS gender,
        'Unknown' AS country,
        CAST(NULL AS DATE) AS signup_date,
        CAST(NULL AS DATE) AS end_date,
        CAST(TRUE AS BOOLEAN) AS is_current
""")

df_dim_customer = df_dim_customer.union(unknown_customer)

print(f"Raw customer rows: {df_dim_customer.count()}")
write_to_gold(df_dim_customer, "dim_customer")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### DIM_PRODUCT

In [ ]:
w = Window.orderBy("product_id")

df_dim_product = (silver_products
    .withColumn("product_key", F.row_number().over(w))
    .withColumn("effective_price_start_date", F.current_date())
    .withColumn("effective_price_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .select(
        "product_key",
        "product_id",
        "product_name",
        "category",
        "brand",
        "price",
        "effective_price_start_date",
        "effective_price_end_date",
        "is_current"
    )
)

# Add unknown products with -1 
unknown_product = spark.sql("""
    SELECT 
        CAST(-1 AS INT) AS product_key,
        CAST(-1 AS INT) AS product_id,
        'Unknown' AS product_name,
        'Unknown' AS category,
        'Unknown' AS brand,
        CAST(NULL AS DECIMAL(10,2)) AS price,
        CAST(NULL AS DATE) AS effective_price_start_date,
        CAST(NULL AS DATE) AS effective_price_end_date,
        CAST(TRUE AS BOOLEAN) AS is_current
""")

df_dim_product = df_dim_product.union(unknown_product)

print(f"Raw product rows: {df_dim_product.count()}")
write_to_gold(df_dim_product, "dim_product")

### FACT_SALES

In [ ]:
#  JOIN orders + order_items 
df_sales_base = (silver_order_items
    .join(silver_orders, on="order_id", how="inner")
)

#  LOOKUP customer_key (fill unmatched keys with -1) 
df_sales_base = (df_sales_base
    .join(
        df_dim_customer.filter(F.col("customer_key") != -1)
            .select("customer_key", "customer_id"),
        on="customer_id",
        how="left"
    )
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), F.lit(-1))
         .otherwise(F.col("customer_key"))
    )
)

#  LOOKUP product_key (fill unmatched keys with -1) 
df_sales_base = (df_sales_base
    .join(
        df_dim_product.filter(F.col("product_key") != -1)
            .select("product_key", "product_id"),
        on="product_id",
        how="left"
    )
    .withColumn("product_key",
        F.when(F.col("product_key").isNull(), F.lit(-1))
         .otherwise(F.col("product_key"))
    )
)

#  CREATE order_date_key 
df_fact_sales = (df_sales_base
    .withColumn("order_date_key",
        F.date_format(F.col("order_date"), "yyyyMMdd").cast(IntegerType())
    )
    .select(
        "order_item_id",
        "customer_key",
        "product_key",
        "order_date_key",
        "order_id",
        "quantity",
        "unit_price",
        "line_total",
        "payment_method",
        "shipping_country"
    )
    .withColumnRenamed("line_total", "total_amount")
)

print(f"Raw sales rows: {df_fact_sales.count()}")
write_to_gold(df_fact_sales, "fact_sales")

### FACT_REVIEWS

In [40]:
# LOOKUP customer_key (fill unmatched keys with -1) 
df_reviews_base = (silver_product_reviews
    .join(
        df_dim_customer.filter(F.col("customer_key") != -1)
            .select("customer_key", "customer_id"),
        on="customer_id",
        how="left"
    )
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), F.lit(-1))
         .otherwise(F.col("customer_key"))
    )
)

#  LOOKUP product_key (fill unmatched keys with -1)---
df_reviews_base = (df_reviews_base
    .join(
        df_dim_product.filter(F.col("product_key") != -1)
            .select("product_key", "product_id"),
        on="product_id",
        how="left"
    )
    .withColumn("product_key",
        F.when(F.col("product_key").isNull(), F.lit(-1))
         .otherwise(F.col("product_key"))
    )
)

#  CREATE review_date_key 
df_fact_reviews = (df_reviews_base
    .withColumn("review_date_key",
        F.date_format(F.col("review_date"), "yyyyMMdd").cast(IntegerType())
    )
    .select(
        "review_id",
        "customer_key",
        "product_key",
        "review_date_key",
        "rating",
        "review_text"
    )
)

print(f"Raw review rows: {df_fact_reviews.count()}")
write_to_gold(df_fact_reviews, "fact_reviews")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 42, Finished, Available, Finished, False)

Raw review rows: 4000000
✅ fact_reviews → 4000000 rows written to Gold


### AGG_CUSTOMER_360

In [41]:
#  SALES METRICS PER CUSTOMER 
customer_sales = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date"), 
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("customer_key")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_bought"),
        F.sum("total_amount").alias("total_lifetime_spend"),
        F.min("full_date").alias("first_order_date"),
        F.max("full_date").alias("last_order_date")
    )
    .withColumn("avg_order_value",
        F.round(F.col("total_lifetime_spend") / F.col("total_orders"), 2)
    )
    .withColumn("days_since_last_order",
        F.datediff(F.current_date(), F.col("last_order_date"))
    )
)

#  REVIEW METRICS PER CUSTOMER 
customer_reviews = (df_fact_reviews
    .groupBy("customer_key")
    .agg(
        F.count("review_id").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating_given")
    )
)

#  COMBINE SALES + REVIEWS + CUSTOMER INFO 
df_agg_customer = (df_dim_customer
    .join(customer_sales, on="customer_key", how="left")
    .join(customer_reviews, on="customer_key", how="left")
    .fillna(0, subset=[
        "total_orders", "total_items_bought", "total_lifetime_spend",
        "total_reviews"
    ])
)

#  CUSTOMER SEGMENT (percentile-based) 
w_spend = Window.orderBy(F.col("total_lifetime_spend").desc())
df_agg_customer = df_agg_customer.withColumn(
    "spend_rank_pct",
    F.percent_rank().over(w_spend)
)

df_agg_customer = df_agg_customer.withColumn(
    "customer_segment",
    F.when(F.col("total_orders") == 0, F.lit("Inactive"))
     .when(F.col("spend_rank_pct") <= 0.10, F.lit("Platinum"))
     .when(F.col("spend_rank_pct") <= 0.30, F.lit("Gold"))
     .when(F.col("spend_rank_pct") <= 0.60, F.lit("Silver"))
     .otherwise(F.lit("Bronze"))
)

#  CUSTOMER STATUS (recency-based) 
df_agg_customer = df_agg_customer.withColumn(
    "customer_status",
    F.when(F.col("total_orders") == 0, F.lit("Never Purchased"))
     .when(F.col("days_since_last_order") <= 90, F.lit("Active"))
     .when(F.col("days_since_last_order") <= 180, F.lit("At Risk"))
     .otherwise(F.lit("Churned"))
)

#  FINAL SELECT 
df_agg_customer_360 = df_agg_customer.select(
    "customer_key",
    "customer_id",
    "name",
    "country",
    "gender",
    "signup_date",
    "total_orders",
    "total_items_bought",
    F.round("total_lifetime_spend", 2).alias("total_lifetime_spend"),
    F.round("avg_order_value", 2).alias("avg_order_value"),
    "first_order_date",
    "last_order_date",
    "days_since_last_order",
    "total_reviews",
    "avg_rating_given",
    "customer_segment",
    "customer_status"
)

print(f"Raw customer 360 rows: {df_agg_customer_360.count()}")
write_to_gold(df_agg_customer_360, "agg_customer_360")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 43, Finished, Available, Finished, False)

Raw customer 360 rows: 1048576
✅ agg_customer_360 → 1048576 rows written to Gold


### AGG_PRODUCT_SCORECARD

In [42]:
#  SALES METRICS PER PRODUCT 
product_sales = (df_fact_sales
    .groupBy("product_key")
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_key").alias("unique_buyers")
    )
)

#  REVIEW METRICS PER PRODUCT 
product_reviews = (df_fact_reviews
    .groupBy("product_key")
    .agg(
        F.count("review_id").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
)

#  COMBINE 
df_agg_product = (df_dim_product
    .select("product_key", "product_id", "product_name", "category", "brand", "price")
    .join(product_sales, on="product_key", how="left")
    .join(product_reviews, on="product_key", how="left")
    .fillna(0, subset=["total_units_sold", "total_revenue", "total_orders", 
                        "unique_buyers", "total_reviews"])
    .fillna(0.0, subset=["avg_rating"])
)

#  REVIEW TO PURCHASE RATIO 
df_agg_product = df_agg_product.withColumn(
    "review_to_purchase_ratio",
    F.when(F.col("unique_buyers") > 0,
        F.round(F.col("total_reviews") / F.col("unique_buyers"), 2)
    ).otherwise(F.lit(0.0))
)

#  PRODUCT TIER (BCG Matrix) 
# High/Low revenue = above/below median revenue
# High/Low rating = above/below 3.5

median_revenue = df_agg_product.filter(F.col("total_revenue") > 0) \
    .approxQuantile("total_revenue", [0.5], 0.01)[0]

df_agg_product = df_agg_product.withColumn(
    "product_tier",
    F.when(F.col("total_revenue") == 0, F.lit("No Sales"))
     .when((F.col("total_revenue") >= median_revenue) & (F.col("avg_rating") > 3.5), 
           F.lit("⭐ Star Product"))
     .when((F.col("total_revenue") >= median_revenue) & (F.col("avg_rating") <= 3.5), 
           F.lit("🐄 Cash Cow"))
     .when((F.col("total_revenue") < median_revenue) & (F.col("avg_rating") > 3.5), 
           F.lit("💎 Hidden Gem"))
     .otherwise(F.lit("❌ Underperformer"))
)

#  FINAL SELECT 
df_agg_product_scorecard = df_agg_product.select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "brand",
    "price",
    "total_units_sold",
    F.round("total_revenue", 2).alias("total_revenue"),
    "total_orders",
    "unique_buyers",
    "total_reviews",
    "avg_rating",
    "review_to_purchase_ratio",
    "product_tier"
)

print(f"Raw product scorecard rows: {df_agg_product_scorecard.count()}")
write_to_gold(df_agg_product_scorecard, "agg_product_scorecard")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 44, Finished, Available, Finished, False)

Raw product scorecard rows: 20001
✅ agg_product_scorecard → 20001 rows written to Gold


### AGG_GEOGRAPHIC_PERFORMANCE

In [43]:
#  BASE: revenue + orders per country 
geo_base = (df_fact_sales
    .groupBy("shipping_country")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("customer_key").alias("total_customers"),
        F.sum("quantity").alias("total_items_sold")
    )
    .withColumn("avg_order_value",
        F.round(F.col("total_revenue") / F.col("total_orders"), 2)
    )
)

#  TOP CATEGORY PER COUNTRY 
category_revenue = (df_fact_sales
    .join(df_dim_product.select("product_key", "category"), on="product_key", how="inner")
    .groupBy("shipping_country", "category")
    .agg(F.sum("total_amount").alias("cat_revenue"))
)

w_cat = Window.partitionBy("shipping_country").orderBy(F.col("cat_revenue").desc())
top_category = (category_revenue
    .withColumn("rn", F.row_number().over(w_cat))
    .filter(F.col("rn") == 1)
    .select("shipping_country", F.col("category").alias("top_category"))
)

#  TOP BRAND PER COUNTRY 
brand_revenue = (df_fact_sales
    .join(df_dim_product.select("product_key", "brand"), on="product_key", how="inner")
    .groupBy("shipping_country", "brand")
    .agg(F.sum("total_amount").alias("brand_revenue"))
)

w_brand = Window.partitionBy("shipping_country").orderBy(F.col("brand_revenue").desc())
top_brand = (brand_revenue
    .withColumn("rn", F.row_number().over(w_brand))
    .filter(F.col("rn") == 1)
    .select("shipping_country", F.col("brand").alias("top_brand"))
)

#  TOP PAYMENT METHOD PER COUNTRY 
payment_counts = (df_fact_sales
    .groupBy("shipping_country", "payment_method")
    .agg(F.countDistinct("order_id").alias("pm_orders"))
)

w_pm = Window.partitionBy("shipping_country").orderBy(F.col("pm_orders").desc())
top_payment = (payment_counts
    .withColumn("rn", F.row_number().over(w_pm))
    .filter(F.col("rn") == 1)
    .select("shipping_country", 
            F.col("payment_method").alias("top_payment_method"),
            F.col("pm_orders").alias("top_payment_orders"))
)

#  AVG PRODUCT RATING PER COUNTRY 
geo_ratings = (df_fact_reviews
    .join(df_dim_customer.select("customer_key", "country"), on="customer_key", how="inner")
    .groupBy("country")
    .agg(F.round(F.avg("rating"), 2).alias("avg_product_rating"))
)

#  COMBINE EVERYTHING 
df_agg_geo = (geo_base
    .join(top_category, on="shipping_country", how="left")
    .join(top_brand, on="shipping_country", how="left")
    .join(top_payment, on="shipping_country", how="left")
    .join(geo_ratings, geo_base.shipping_country == geo_ratings.country, "left")
    .drop("country")
)

#  REVENUE RANK 
df_agg_geo = df_agg_geo.withColumn(
    "revenue_rank",
    F.row_number().over(Window.orderBy(F.col("total_revenue").desc()))
)

#  FINAL SELECT 
df_agg_geographic = df_agg_geo.select(
    "shipping_country",
    "total_orders",
    F.round("total_revenue", 2).alias("total_revenue"),
    "total_customers",
    "avg_order_value",
    "total_items_sold",
    "top_category",
    "top_brand",
    "top_payment_method",
    "avg_product_rating",
    "revenue_rank"
)

print(f"Raw geographic rows: {df_agg_geographic.count()}")
write_to_gold(df_agg_geographic, "agg_geographic_performance")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 45, Finished, Available, Finished, False)

Raw geographic rows: 243
✅ agg_geographic_performance → 243 rows written to Gold


### AGG_MONTHLY_REVENUE_TRENDS

In [44]:
#  BASE: monthly sales metrics 
monthly_base = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number", "month_name"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("year", "month_number", "month_name")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_sold"),
        F.countDistinct("customer_key").alias("unique_customers")
    )
    .withColumn("avg_order_value",
        F.round(F.col("total_revenue") / F.col("total_orders"), 2)
    )
)

#  NEW CUSTOMERS PER MONTH
customer_first_order = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("customer_key")
    .agg(F.min("full_date").alias("first_order_date"))
    .withColumn("cohort_year", F.year("first_order_date"))
    .withColumn("cohort_month", F.month("first_order_date"))
)

new_customers_monthly = (customer_first_order
    .groupBy(
        F.col("cohort_year").alias("year"),
        F.col("cohort_month").alias("month_number")
    )
    .agg(F.count("customer_key").alias("new_customers"))
)

#  RETURNING CUSTOMERS PER MONTH 
all_customers_monthly = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .select("customer_key", "year", "month_number")
    .distinct()
)

returning_monthly = (all_customers_monthly
    .join(customer_first_order, on="customer_key", how="inner")
    .filter(
        (F.col("year") != F.col("cohort_year")) |
        (F.col("month_number") != F.col("cohort_month"))
    )
    .groupBy("year", "month_number")
    .agg(F.countDistinct("customer_key").alias("returning_customers"))
)

#  COMBINE 
df_agg_monthly = (monthly_base
    .join(new_customers_monthly, on=["year", "month_number"], how="left")
    .join(returning_monthly, on=["year", "month_number"], how="left")
    .fillna(0, subset=["new_customers", "returning_customers"])
)

#  RETENTION RATE 
df_agg_monthly = df_agg_monthly.withColumn(
    "retention_rate",
    F.when(F.col("unique_customers") > 0,
        F.round(F.col("returning_customers") / F.col("unique_customers") * 100, 2)
    ).otherwise(F.lit(0.0))
)

#  REVENUE GROWTH % (month over month) 
w_month = Window.orderBy("year", "month_number")
df_agg_monthly = df_agg_monthly.withColumn(
    "prev_month_revenue", F.lag("total_revenue").over(w_month)
)

df_agg_monthly = df_agg_monthly.withColumn(
    "revenue_growth_pct",
    F.when(F.col("prev_month_revenue").isNotNull() & (F.col("prev_month_revenue") > 0),
        F.round(
            (F.col("total_revenue") - F.col("prev_month_revenue")) 
            / F.col("prev_month_revenue") * 100, 2
        )
    ).otherwise(F.lit(None))
)

#  TOP CATEGORY PER MONTH 
monthly_category = (df_fact_sales
    .join(df_dim_date.select("date_key", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .join(df_dim_product.select("product_key", "category"), on="product_key", how="inner")
    .groupBy("year", "month_number", "category")
    .agg(F.sum("total_amount").alias("cat_rev"))
)

w_mcat = Window.partitionBy("year", "month_number").orderBy(F.col("cat_rev").desc())
top_monthly_cat = (monthly_category
    .withColumn("rn", F.row_number().over(w_mcat))
    .filter(F.col("rn") == 1)
    .select("year", "month_number", F.col("category").alias("top_category"))
)

#  FINAL COMBINE 
df_agg_monthly = df_agg_monthly.join(top_monthly_cat, on=["year", "month_number"], how="left")

#  FINAL SELECT 
df_agg_monthly_trends = df_agg_monthly.select(
    "year",
    "month_number",
    "month_name",
    F.round("total_revenue", 2).alias("total_revenue"),
    "total_orders",
    "total_items_sold",
    "avg_order_value",
    "unique_customers",
    "new_customers",
    "returning_customers",
    "retention_rate",
    "revenue_growth_pct",
    "top_category"
).orderBy("year", "month_number")

print(f"Raw monthly trend rows: {df_agg_monthly_trends.count()}")
write_to_gold(df_agg_monthly_trends, "agg_monthly_revenue_trends")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 46, Finished, Available, Finished, False)

Raw monthly trend rows: 49
✅ agg_monthly_revenue_trends → 49 rows written to Gold


### Final data Validation

In [45]:
print("GOLD LAYER VERIFICATION")

gold_tables = [
    "dim_date", "dim_customer", "dim_product",
    "fact_sales", "fact_reviews",
    "agg_customer_360", "agg_product_scorecard",
    "agg_geographic_performance", "agg_monthly_revenue_trends"
]

for table in gold_tables:
    try:
        df = spark.read.format("delta").table(table)
        print(f"\n {table.upper()} — {df.count():,} rows, {len(df.columns)} cols")
        for field in df.schema.fields:
            nullable = "nullable" if field.nullable else "not null"
            print(f"   ├── {field.name:<30} {str(field.dataType):<20} {nullable}")
    except Exception as e:
        print(f"\n {table.upper()} — Error: {str(e)}")

print("  Gold Layer Completed.")

StatementMeta(, 0840f5ff-83d9-4484-9e81-b13070612aec, 47, Finished, Available, Finished, False)

GOLD LAYER VERIFICATION

 DIM_DATE — 1,461 rows, 9 cols
   ├── date_key                       IntegerType()        nullable
   ├── full_date                      DateType()           nullable
   ├── year                           IntegerType()        nullable
   ├── quarter                        IntegerType()        nullable
   ├── month_number                   IntegerType()        nullable
   ├── month_name                     StringType()         nullable
   ├── day_of_week                    IntegerType()        nullable
   ├── day_name                       StringType()         nullable
   ├── is_weekend                     BooleanType()        nullable

 DIM_CUSTOMER — 1,048,576 rows, 9 cols
   ├── customer_key                   IntegerType()        nullable
   ├── customer_id                    IntegerType()        nullable
   ├── name                           StringType()         nullable
   ├── email                          StringType()         nullable
   ├── gender       